# 02 Run Experiment

            Generic inference runner. Change only the config cell to run knowledge, reasoning, context, or robustness jobs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import random
import re
import string
import time

def env_int(name, default):
    value = os.getenv(name)
    return default if value in (None, "") else int(value)


def env_optional_int(name, default):
    value = os.getenv(name)
    if value in (None, ""):
        return default
    return None if value.lower() == "none" else int(value)


def env_bool(name, default):
    value = os.getenv(name)
    if value in (None, ""):
        return default
    return value.lower() in {"1", "true", "yes", "y"}


CORE_DATA_FILES = ["knowledge_data.jsonl", "reasoning_gsm8k.jsonl", "context_hotpotqa.jsonl"]


def has_core_data_files(path):
    return all((path / name).exists() for name in CORE_DATA_FILES)


def candidate_data_dirs():
    yield Path("/kaggle/input/notebooks/nguynnguynhehe/00-prepare-data/slm_limits_data")
    notebooks_root = Path("/kaggle/input/notebooks")
    if notebooks_root.exists():
        yield from sorted(notebooks_root.glob("*/00-prepare-data/slm_limits_data"))
    yield Path("/kaggle/input/slm-limits-data")
    input_root = Path("/kaggle/input")
    if input_root.exists():
        yield from sorted(input_root.glob("*/slm_limits_data"))
    yield Path("slm_limits_data")


def resolve_data_dir(value=None):
    if value:
        return Path(value)
    for candidate in candidate_data_dirs():
        if has_core_data_files(candidate):
            return candidate
    return Path("slm_limits_data")


AXIS = os.getenv("AXIS", "robustness")
MODEL = os.getenv("MODEL", "Qwen2.5-3B")
MODEL_NAME_MAP = {
    "Qwen2.5-1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "Qwen2.5-7B": "Qwen/Qwen2.5-7B-Instruct",
}
MODEL_NAME = MODEL_NAME_MAP.get(MODEL, MODEL)
MODEL_SIZE = MODEL.replace("Qwen/Qwen2.5-", "").replace("Qwen2.5-", "").replace("-Instruct", "")
CONDITION = os.getenv("CONDITION", "robust_original")
CONDITIONS_TO_RUN = [condition.strip() for condition in os.getenv("CONDITIONS_TO_RUN", "robust_para,robust_typo,robust_format").split(",") if condition.strip()]
MAX_SAMPLES = env_optional_int("MAX_SAMPLES", 1000)
SHARD_ID = env_int("SHARD_ID", 0)
NUM_SHARDS = env_int("NUM_SHARDS", 4)
SEED = env_int("SEED", 42)
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NEW_TOKENS = env_int("MAX_NEW_TOKENS", 384)
BACKEND = os.getenv("BACKEND", "transformers")
LOCAL_FILES_ONLY = env_bool("LOCAL_FILES_ONLY", False)
REQUIRE_REAL_DATA = env_bool("REQUIRE_REAL_DATA", True)
OVERWRITE_OUTPUT = env_bool("OVERWRITE_OUTPUT", False)
ROBUSTNESS_DATASET = os.getenv("ROBUSTNESS_DATASET", "gsm8k")

DATA_DIR = resolve_data_dir(os.getenv("DATA_DIR"))
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "outputs"))
PROMPT_TEMPLATE = "experiment_v3_robustness_fix"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)


In [ ]:
AXIS_TO_DATA_PATH = {
    "knowledge": DATA_DIR / "knowledge_data.jsonl",
    "reasoning": DATA_DIR / "reasoning_gsm8k.jsonl",
    "context": DATA_DIR / "context_hotpotqa.jsonl",
    "robustness": DATA_DIR / f"robustness_{ROBUSTNESS_DATASET}.jsonl",
}

AXIS_CONDITIONS = {
    "robustness": {"robust_original", "robust_para", "robust_typo", "robust_format"}
}


def model_tag(value):
    tag = value.split("/")[-1].replace("-Instruct", "")
    return tag.replace(".", "p").replace("-", "_")


def output_path():
    sample_tag = "full" if MAX_SAMPLES is None else f"n{MAX_SAMPLES}"
    name = f"records_experiment_{AXIS}_{model_tag(MODEL_NAME)}_{CONDITION}_{sample_tag}_shard{SHARD_ID}.jsonl"
    return OUTPUT_DIR / name


DATA_PATH = AXIS_TO_DATA_PATH.get(AXIS)
OUTPUT_PATH = output_path()


def validate_config():
    if AXIS not in AXIS_TO_DATA_PATH or AXIS not in AXIS_CONDITIONS:
        allowed_axes = sorted(set(AXIS_TO_DATA_PATH) & set(AXIS_CONDITIONS))
        raise ValueError(f"Unsupported AXIS: {AXIS}. Choose one of {allowed_axes}.")
    allowed = sorted(AXIS_CONDITIONS[AXIS])
    if CONDITION not in AXIS_CONDITIONS[AXIS]:
        raise ValueError(f"Unsupported CONDITION for {AXIS}: {CONDITION}. Choose one of {allowed}.")
    unsupported_conditions = [condition for condition in CONDITIONS_TO_RUN if condition not in AXIS_CONDITIONS[AXIS]]
    if unsupported_conditions:
        raise ValueError(f"Unsupported CONDITIONS_TO_RUN for {AXIS}: {unsupported_conditions}. Choose from {allowed}.")
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Missing data file: {DATA_PATH}. Run 00-prepare-data.ipynb first.")
    if not (0 <= SHARD_ID < NUM_SHARDS):
        raise ValueError("SHARD_ID must satisfy 0 <= SHARD_ID < NUM_SHARDS.")


validate_config()

In [ ]:
def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def validate_real_rows(rows):
    if not rows:
        raise ValueError(f"{DATA_PATH} has no rows.")
    if not REQUIRE_REAL_DATA:
        return
    fallback_count = sum(1 for row in rows if row.get("metadata", {}).get("source") == "local_fallback")
    if fallback_count == len(rows):
        raise ValueError(f"{DATA_PATH} looks like local fallback/toy data. Run 00 with HuggingFace data first.")


def append_jsonl(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def successful_keys(path):
    if OVERWRITE_OUTPUT and path.exists():
        path.unlink()
        return set()
    keys = set()
    if not path.exists():
        return keys
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if row.get("status") == "success":
                keys.add(row.get("unique_key"))
    return keys


def select_samples(rows):
    selected = rows if MAX_SAMPLES is None else rows[:MAX_SAMPLES]
    return [row for index, row in enumerate(selected) if index % NUM_SHARDS == SHARD_ID]


def unique_key(sample):
    parts = [MODEL_NAME, AXIS, sample["dataset"], CONDITION, sample["sample_id"], PROMPT_TEMPLATE]
    return "::".join(str(part) for part in parts)

In [ ]:
ARTICLES = {"a", "an", "the"}


def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [token for token in text.split() if token not in ARTICLES]
    return " ".join(tokens)


def token_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    overlap = 0
    remaining = gold_tokens.copy()
    for token in pred_tokens:
        if token in remaining:
            overlap += 1
            remaining.remove(token)
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def has_final_answer(text):
    return re.search(r"final answer\s*:", str(text), flags=re.IGNORECASE) is not None


def extract_after_final_answer(text):
    match = re.search(r"final answer\s*:\s*(.*)", str(text), flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(1)
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    return lines[-1] if lines else ""


def clean_short_qa_answer(answer):
    answer = str(answer).strip()
    answer = re.split(
        r"\.\s+(?:To determine|This|Therefore|Based on|I |The |It |We )",
        answer,
        maxsplit=1,
    )[0]
    answer = re.sub(r"^(the answer is|answer is|answer)\s*:?\s*", "", answer, flags=re.IGNORECASE)
    return answer.strip(" .")


def extract_short_qa_answer(text):
    match = re.search(r"final answer\s*:\s*(.*)", str(text), flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(1)
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    if not lines:
        return ""
    return clean_short_qa_answer(lines[0])


def extract_json_answer(text):
    try:
        parsed = json.loads(str(text).strip())
    except json.JSONDecodeError:
        return ""
    return str(parsed.get("answer", "")) if isinstance(parsed, dict) else ""


def extract_numeric(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    if not numbers:
        return ""
    value = numbers[-1]
    return value[:-2] if value.endswith(".0") else value


def extract_answer(raw_text, axis, condition):
    if condition.endswith("_format"):
        parsed = extract_json_answer(raw_text)
        if parsed:
            return parsed
    if axis == "reasoning" or (axis == "robustness" and ROBUSTNESS_DATASET == "gsm8k"):
        text = extract_after_final_answer(raw_text) if has_final_answer(raw_text) else raw_text
        numeric = extract_numeric(text)
        return numeric or extract_after_final_answer(raw_text)
    return extract_short_qa_answer(raw_text)


def score_prediction(prediction, gold, sample):
    if sample["dataset"] == "gsm8k":
        pred_num = extract_numeric(prediction)
        gold_num = extract_numeric(gold)
        accuracy = float(pred_num != "" and pred_num == gold_num)
        return accuracy, accuracy, accuracy
    exact_match = float(normalize_text(prediction) == normalize_text(gold))
    f1 = token_f1(prediction, gold)
    return exact_match, f1, exact_match

In [ ]:
def context_lines(sample):
    return [line.strip() for line in str(sample.get("context", "")).splitlines() if line.strip()]


def supporting_titles(sample):
    facts = sample.get("supporting_facts", {})
    if isinstance(facts, dict):
        return {str(title) for title in facts.get("title", [])}
    return set()


def split_hotpot_context(sample):
    lines = context_lines(sample)
    titles = supporting_titles(sample)
    if not lines or not titles:
        return "\n".join(lines), ""
    oracle = []
    distractors = []
    for line in lines:
        title = line.split(":", 1)[0]
        if title in titles:
            oracle.append(line)
        else:
            distractors.append(line)
    return "\n\n".join(oracle or lines), "\n\n".join(distractors)


def paragraph_title(paragraph):
    return str(paragraph).split(":", 1)[0].strip()


def split_hotpot_context_parts(sample):
    paragraphs = context_lines(sample)
    titles = supporting_titles(sample)
    if not paragraphs:
        return [], []
    if not titles:
        return paragraphs, []
    oracle = []
    distractors = []
    for paragraph in paragraphs:
        if paragraph_title(paragraph) in titles:
            oracle.append(paragraph)
        else:
            distractors.append(paragraph)
    return oracle or paragraphs, distractors


def bm25_like_top_paragraphs(sample, k=5):
    paragraphs = context_lines(sample)
    if len(paragraphs) <= k:
        return paragraphs
    query_terms = normalize_text(sample.get("question", "")).split()
    if not query_terms:
        return paragraphs[:k]
    scored = []
    for index, paragraph in enumerate(paragraphs):
        terms = normalize_text(paragraph).split()
        if not terms:
            scored.append((0.0, index, paragraph))
            continue
        term_counts = {term: terms.count(term) for term in set(terms)}
        score = sum(term_counts.get(term, 0) / len(terms) for term in query_terms)
        scored.append((score, index, paragraph))
    return [paragraph for _, _, paragraph in sorted(scored, key=lambda item: (-item[0], item[1]))[:k]]


def gold_position(selected_paragraphs, sample):
    titles = supporting_titles(sample)
    if not selected_paragraphs or not titles:
        return "none"
    positions = [index for index, paragraph in enumerate(selected_paragraphs) if paragraph_title(paragraph) in titles]
    if not positions:
        return "none"
    if positions == list(range(len(positions))):
        return "beginning"
    if positions == list(range(len(selected_paragraphs) - len(positions), len(selected_paragraphs))):
        return "end"
    return "mixed"


def context_selection(sample):
    oracle, distractors = split_hotpot_context_parts(sample)
    first_distractors = distractors[:3]
    source = "none"
    selected = []
    selected_distractors = []

    if CONDITION == "context_retrieved":
        selected = bm25_like_top_paragraphs(sample, k=5)
        selected_distractors = [p for p in selected if paragraph_title(p) not in supporting_titles(sample)]
        source = "bm25"
    elif CONDITION == "context_oracle":
        selected = oracle
        source = "oracle"
    elif CONDITION == "context_oracle_distractor":
        selected = oracle + first_distractors
        selected_distractors = first_distractors
        source = "oracle+distractor"
    elif CONDITION == "context_oracle_end":
        selected = first_distractors + oracle
        selected_distractors = first_distractors
        source = "oracle_end"

    text = "\n\n".join(selected)
    distractor_text = "\n\n".join(selected_distractors)
    gold_answer = normalize_text(sample.get("gold_answer", ""))
    normalized_distractors = normalize_text(distractor_text)
    return {
        "text": text,
        "num_context_paragraphs": len(selected),
        "num_oracle_paragraphs": sum(1 for paragraph in selected if paragraph_title(paragraph) in supporting_titles(sample)),
        "num_distractor_paragraphs": len(selected_distractors),
        "context_tokens": len(text.split()),
        "gold_at_position": gold_position(selected, sample),
        "context_source": source,
        "context_truncated": False,
        "distractor_contains_answer": bool(gold_answer and gold_answer in normalized_distractors),
    }


def empty_context_metadata():
    return {
        "num_context_paragraphs": 0,
        "num_oracle_paragraphs": 0,
        "num_distractor_paragraphs": 0,
        "context_tokens": 0,
        "gold_at_position": "none",
        "context_source": "none",
        "context_truncated": False,
        "distractor_contains_answer": False,
    }


def record_context_metadata(sample):
    if AXIS != "context":
        return empty_context_metadata() | {"text": sample.get("context", "")}
    return context_selection(sample)


def sample_index(sample):
    numbers = re.findall(r"\d+", str(sample.get("sample_id", "")))
    return int(numbers[-1]) if numbers else 0


def knowledge_context(sample, all_rows, include_distractors=False):
    context = str(sample.get("context", "")).strip()
    if not context:
        context = f"Reference: The answer is {sample['gold_answer']}."
    if not include_distractors:
        return context
    others = [row for row in all_rows if row.get("sample_id") != sample.get("sample_id")]
    distractors = []
    for row in others[:3]:
        distractors.append(f"Distractor: The answer to another question is {row.get('gold_answer', '')}.")
    return context + "\n" + "\n".join(distractors)


def oracle_step(sample):
    answer = str(sample.get("gold_answer", "")).strip()
    return f"The final numeric answer should be {answer}."


TYPO_REPLACEMENTS = {
    "answer": "anwser",
    "question": "quesiton",
    "problem": "probelm",
    "calculate": "calcluate",
    "total": "toatl",
    "years": "yaers",
    "large": "lagre",
    "small": "smal",
    "many": "mnay",
    "does": "deos",
    "have": "haev",
    "trades": "traeds",
}


def typo_text(text):
    def replace_word(match):
        word = match.group(0)
        typo = TYPO_REPLACEMENTS.get(word.lower())
        return typo.capitalize() if typo and word[0].isupper() else typo or word

    original = str(text)
    typo = re.sub(r"\b[A-Za-z]+\b", replace_word, original)
    if typo != original:
        return typo

    return re.sub(r"\b([A-Za-z])([A-Za-z])([A-Za-z]{2,})\b", r"\2\1\3", original, count=1)


def build_prompt(sample, all_rows):
    question = sample["question"]
    context = str(sample.get("context", "")).strip()

    if CONDITION == "knowledge_closed":
        return f"Answer with a short phrase only. Do not explain.\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "knowledge_oracle":
        context = knowledge_context(sample, all_rows)
        return f"Answer with a short phrase only. Do not explain. Use the context when possible.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "knowledge_distractor":
        context = knowledge_context(sample, all_rows, include_distractors=True)
        return f"Answer with a short phrase only. Do not explain. Some context may be irrelevant.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"

    if CONDITION == "reasoning_direct":
        return f"Solve the math problem. Return only one number. Do not explain.\n\nProblem: {question}\n\nFinal answer:"
    if CONDITION == "reasoning_cot":
        return f"Solve the math problem step by step. End with \"Final answer: <number>\".\n\nProblem: {question}\n\nFinal answer:"
    if CONDITION == "reasoning_oracle_step":
        return f"Problem: {question}\n\nHelpful intermediate step: {oracle_step(sample)}\n\nContinue and give only the final numeric answer.\n\nFinal answer:"

    if CONDITION == "context_none":
        return f"Answer with a short phrase only. Do not explain.\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "context_retrieved":
        context = context_selection(sample)["text"]
        return f"Answer with a short phrase only. Do not explain. Use the context when possible.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "context_oracle":
        context = context_selection(sample)["text"]
        return f"Answer with a short phrase only. Do not explain. Use the context when possible.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "context_oracle_distractor":
        context = context_selection(sample)["text"]
        return f"Answer with a short phrase only. Do not explain. Some context may be irrelevant.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "context_oracle_end":
        context = context_selection(sample)["text"]
        return f"Answer with a short phrase only. Do not explain. Some context may be irrelevant.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"

    if CONDITION == "robust_original":
        if sample["dataset"] == "gsm8k":
            return f"Solve the math problem. Return only one number. Do not explain.\n\nProblem: {question}\n\nFinal answer:"
        return f"Answer with a short phrase only. Do not explain.\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "robust_para":
        if sample["dataset"] == "gsm8k":
            return f"Find the final result for this word problem. Reply with just the number.\n\nProblem: {question}\n\nFinal answer:"
        return f"Give the concise answer to the question below without explanation.\n\nQuestion: {question}\n\nFinal answer:"
    if CONDITION == "robust_typo":
        typo_question = typo_text(question)
        if sample["dataset"] == "gsm8k":
            return f"Solve the math problem. Return only one number. Do not explain.\n\nProblem: {typo_question}\n\nFinal answer:"
        return f"Answer with a short phrase only. Do not explain.\n\nQuestion: {typo_question}\n\nFinal answer:"
    if CONDITION == "robust_format":
        if sample["dataset"] == "gsm8k":
            return f'Return JSON only in this exact schema: {{"answer": "<number>"}}\n\nProblem: {question}'
        return f'Return JSON only in this exact schema: {{"answer": "..."}}\n\nQuestion: {question}'

    raise ValueError(f"Unknown condition: {CONDITION}")

In [ ]:
def mock_generate(sample, prompt):
    gold = sample["gold_answer"]
    if CONDITION == "robust_format":
        return json.dumps({"answer": gold})
    if CONDITION.endswith("_cot"):
        return f"We solve it carefully.\nFinal answer: {gold}"
    if CONDITION.endswith("_closed") or CONDITION.endswith("_none"):
        if sample_index(sample) % 4 == 0:
            return "Final answer: unknown"
    return f"Final answer: {gold}"


def load_transformers_model():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=dtype,
        device_map="auto",
        local_files_only=LOCAL_FILES_ONLY,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model


def transformers_generate(tokenizer, model, prompt):
    import torch

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - start
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    input_tokens = int(inputs["input_ids"].shape[-1])
    output_tokens = int(generated_ids.shape[-1])
    memory_gb = 0.0
    if torch.cuda.is_available():
        memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    return text, latency, input_tokens, output_tokens, memory_gb

In [ ]:
if AXIS == "robustness":
    rows = read_jsonl(DATA_PATH)
    samples = select_samples(rows)
    old_condition = CONDITION

    for cond in ["robust_original", "robust_typo", "robust_para", "robust_format"]:
        CONDITION = cond
        print("\n===", cond, "===")
        for sample in samples[:3]:
            print(build_prompt(sample, rows)[:500])
            print("---")

    CONDITION = "robust_original"
    p_original = [build_prompt(sample, rows) for sample in samples[:20]]

    CONDITION = "robust_typo"
    p_typo = [build_prompt(sample, rows) for sample in samples[:20]]

    same_count = sum(a == b for a, b in zip(p_original, p_typo))
    print("same original vs typo prompts:", same_count, "/", len(p_original))

    for original_prompt, typo_prompt in zip(p_original, p_typo):
        if original_prompt != typo_prompt:
            print("\nExample changed prompt:")
            print("ORIGINAL:\n", original_prompt[:500])
            print("\nTYPO:\n", typo_prompt[:500])
            break

    CONDITION = old_condition

In [ ]:
rows = read_jsonl(DATA_PATH)
validate_real_rows(rows)
samples = select_samples(rows)

model_bundle = load_transformers_model() if BACKEND == "transformers" else None
run_summaries = []
original_condition = CONDITION

for condition in CONDITIONS_TO_RUN:
    CONDITION = condition
    OUTPUT_PATH = output_path()
    completed = successful_keys(OUTPUT_PATH)

    print(
        {
            "axis": AXIS,
            "condition": CONDITION,
            "samples": len(samples),
            "model_name": MODEL_NAME,
            "backend": BACKEND,
            "output_path": str(OUTPUT_PATH),
        }
    )

    records_written = 0

    for sample in samples:
        key = unique_key(sample)
        if key in completed:
            continue

        prompt = build_prompt(sample, rows)
        context_metadata = record_context_metadata(sample)
        start = time.perf_counter()
        status = "success"
        error_message = ""
        input_tokens = len(prompt.split())
        output_tokens = 0
        max_memory_gb = 0.0

        try:
            if BACKEND == "transformers":
                tokenizer, model = model_bundle
                raw, latency, input_tokens, output_tokens, max_memory_gb = transformers_generate(tokenizer, model, prompt)
            else:
                raw = mock_generate(sample, prompt)
                latency = time.perf_counter() - start
                output_tokens = len(raw.split())

            prediction = extract_answer(raw, AXIS, CONDITION)
            exact_match, f1, accuracy = score_prediction(prediction, sample["gold_answer"], sample)
            main_score = accuracy if sample["dataset"] == "gsm8k" else f1
            tokens_per_second = output_tokens / latency if latency > 0 else 0.0
        except Exception as exc:
            raw = ""
            prediction = ""
            exact_match = f1 = accuracy = main_score = 0.0
            latency = time.perf_counter() - start
            tokens_per_second = 0.0
            status = "error"
            error_message = repr(exc)

        record = {
            "unique_key": key,
            "sample_id": sample["sample_id"],
            "axis": AXIS,
            "dataset": sample["dataset"],
            "condition": CONDITION,
            "model_name": MODEL_NAME,
            "model_size": MODEL_SIZE,
            "prompt_template": PROMPT_TEMPLATE,
            "question": sample["question"],
            "context": context_metadata["text"],
            "gold_answer": sample["gold_answer"],
            "prediction_raw": raw,
            "prediction_normalized": prediction,
            "exact_match": exact_match,
            "f1": f1,
            "accuracy": accuracy,
            "main_score": main_score,
            "additional_metrics": {
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "max_new_tokens": MAX_NEW_TOKENS,
                "num_shards": NUM_SHARDS,
                "shard_id": SHARD_ID,
                "robustness_dataset": ROBUSTNESS_DATASET if AXIS == "robustness" else "",
            },
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "latency_seconds": latency,
            "tokens_per_second": tokens_per_second,
            "max_memory_allocated_gb": max_memory_gb,
            "num_context_paragraphs": context_metadata["num_context_paragraphs"],
            "num_oracle_paragraphs": context_metadata["num_oracle_paragraphs"],
            "num_distractor_paragraphs": context_metadata["num_distractor_paragraphs"],
            "context_tokens": context_metadata["context_tokens"],
            "gold_at_position": context_metadata["gold_at_position"],
            "context_source": context_metadata["context_source"],
            "context_truncated": context_metadata["context_truncated"],
            "distractor_contains_answer": context_metadata["distractor_contains_answer"],
            "status": status,
            "error_message": error_message,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }
        append_jsonl(OUTPUT_PATH, record)
        completed.add(key)
        records_written += 1

    summary = {"output_path": str(OUTPUT_PATH), "condition": CONDITION, "records_written": records_written, "total_expected": len(samples)}
    run_summaries.append(summary)
    print(summary)

CONDITION = original_condition
print({"run_summaries": run_summaries})
